# ⚽ Predict the FIFA World Cup 2026

## 📖 Background

The 2026 FIFA World Cup is one of the biggest sporting events in the world, hosted across the United States, Canada, and Mexico. For the first time, the tournament expands to 48 teams, producing 104 matches across the group stage and knockout rounds.

Using machine learning, historical statistics, and soccer domain knowledge, predict match scores, corners, and cards for every fixture. You must submit all your predictions before a single ball is kicked.

The scoring system rewards precision: an exact scoreline earns maximum points, while close predictions still earn partial credit. Later rounds carry score multipliers, so a strong model that holds up in the knockout stages can leapfrog the competition. The challenge is designed to be difficult enough that no one can achieve a perfect score—even with AI assistance—but accessible enough that any data enthusiast can participate and score points.

## 💾 The data

You have access to the following files:

#### `data/group_fixtures.csv` — all 72 group stage matches
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `group` | Group letter (A–L) |
| `home_team` | Home team name |
| `away_team` | Away team name |
| `date` | Match date (UTC) |
| `venue` | Stadium and city |

#### `data/knockout_slots.csv` — all 32 knockout round slots
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `round` | Round name (e.g. `Quarter-final`) |
| `multiplier` | Score multiplier for this round |
| `slot_home` | Description of the home team slot (e.g. `Winner Group A`) |
| `slot_away` | Description of the away team slot |

| Variable | Description |
|---|---|

You may also bring in any external data—FIFA rankings, historical match results, player statistics—to build your predictions.

In [617]:
import pandas as pd

group_fixtures = pd.read_csv('data/group_fixtures.csv')
group_fixtures.head()

,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [618]:
knockout_slots = pd.read_csv('data/knockout_slots.csv')
knockout_slots

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F)
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H)
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I)
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K)
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J)
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J)


## 💪 Competition challenge

The 2026 World Cup has two phases:

- **Group stage** (matches 1–72): The 48 teams are split into 12 groups of 4. Every team plays the other 3 teams in their group once. The best teams from each group advance to the next phase.
- **Knockout stage** (matches 73–104): Single-elimination rounds — Round of 32, Round of 16, Quarter-finals, Semi-finals, and the Final. Lose once and you're out. Crucially, the two teams playing in each knockout match are not known in advance: they depend on who qualified from the group stage.

Submit predictions for **every match** in both phases. For each match you need to predict:

1. **Score** — the exact final scoreline (e.g. `2-1` means the home team scores 2, the away team scores 1). For knockout matches, the score is the result after 90 minutes and extra time — the penalty shootout is not included.
2. **Corners** — the number of corner kicks awarded in the match
3. **Yellow cards** — the number of yellow cards shown in the match
4. **Red cards** — the number of red cards shown in the match

For **group stage** matches, also predict:
- **Winning team** — which team wins the individual match (use `home`, `away`, or `draw`)

For **knockout round** matches, also predict:
- **Matchup** — which two teams you predict will be playing in that slot. Because the bracket is determined by group stage results, you need to predict which teams advance far enough to meet in each round.
- **Match winner** — which team wins the match (use `home` or `away`)
- **Penalties** — whether the match goes to a penalty shootout (`True` or `False`)

### Scoring system

| Category | Condition | Points |
|---|---|---|
| Score | Exact scoreline | 25 |
| Score | Correct goal difference, wrong score | 10 |
| Score | Correct total goals, wrong score | 10 |
| Corners | Exact number | 10 |
| Corners | Off by 2 | 5 |
| Yellow cards | Exact number | 10 |
| Yellow cards | Off by 1 | 5 |
| Red cards | Exact number | 5 |
| Winning team *(group stage only)* | Correct | 40 |
| Matchup *(knockout only)* | Both teams correct | 20 |
| Matchup *(knockout only)* | One team correct | 10 |
| Match winner *(knockout only)* | Correct | 20 |
| Penalties *(knockout only)* | Correct | 5 |

All points for a match are multiplied by the round factor:

| Round | Multiplier |
|---|---|
| Group stage | ×1 |
| Round of 32 | ×1 |
| Round of 16 | ×2 |
| Quarter-final | ×4 |
| Semi-final | ×8 |
| Third-place playoff | ×8 |
| Final | ×16 |

## 🗓️ Group stage predictions

Fill in your predictions for all 72 group stage matches below.

In [619]:
group_predictions = group_fixtures.copy()

# Fill in your predictions for each match
# Example (match 1 — Mexico vs South Africa): predicted_home_goals=2, predicted_away_goals=1, corners=9, yellow_cards=3, red_cards=0, winning_team='home'
group_predictions['predicted_home_goals'] = None   # e.g. 2
group_predictions['predicted_away_goals'] = None   # e.g. 1
group_predictions['corners']              = None   # e.g. 9
group_predictions['yellow_cards']         = None   # e.g. 3
group_predictions['red_cards']            = None   # e.g. 0
group_predictions['winning_team']         = None   # "home", "away", or "draw"

group_predictions

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",None,None,None,None,None,None
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",None,None,None,None,None,None
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",None,None,None,None,None,None
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",None,None,None,None,None,None
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver",None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",None,None,None,None,None,None
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",None,None,None,None,None,None
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",None,None,None,None,None,None
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",None,None,None,None,None,None


## 🏆 Knockout stage predictions

For knockout matches you also predict **which teams are playing**. Fill in the team names based on your group stage predictions, then add your match predictions.

In [620]:
knockout_predictions = knockout_slots.copy()

# Fill in your predictions for each knockout match
# Example (match 73 — Round of 32): predicted_home_team='Brazil', predicted_away_team='France', predicted_home_goals=1, predicted_away_goals=0, corners=8, yellow_cards=2, red_cards=0, match_winner='home', penalties=False
knockout_predictions['predicted_home_team']  = None   # e.g. "Brazil"
knockout_predictions['predicted_away_team']  = None   # e.g. "France"
knockout_predictions['predicted_home_goals'] = None   # e.g. 1
knockout_predictions['predicted_away_goals'] = None   # e.g. 0
knockout_predictions['corners']              = None   # e.g. 8
knockout_predictions['yellow_cards']         = None   # e.g. 2
knockout_predictions['red_cards']            = None   # e.g. 0
knockout_predictions['match_winner']         = None   # "home" or "away"
knockout_predictions['penalties']            = None   # True or False

knockout_predictions

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,None,None,None,None,None,None,None,None,None
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,None,None,None,None,None,None,None,None,None
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),None,None,None,None,None,None,None,None,None
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,None,None,None,None,None,None,None,None,None
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,None,None,None,None,None,None,None,None,None
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),None,None,None,None,None,None,None,None,None
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),None,None,None,None,None,None,None,None,None
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),None,None,None,None,None,None,None,None,None
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),None,None,None,None,None,None,None,None,None
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),None,None,None,None,None,None,None,None,None


## ✅ Checklist before publishing into the competition

- Rename your workspace to make it descriptive of your work. N.B. you should leave the notebook name as `notebook.ipynb`.
- Remove redundant cells like the judging criteria, so the workbook is focused on your predictions.
- Make sure all prediction cells are filled in—`None` values will score 0 points.
- Check that all cells run without error.
- Make sure your workbook is published before **June 10, 2026 at 09:00 UTC**.

## ⏳ Time is ticking. Good luck!

## **Data Exploration & Preprocessing**

1. elo.csv - World Football Elo Ratings
2. history_stat.csv - FIFA World Cup History Statistics
3. former_names.csv

In [621]:
import pandas as pd

# Load historical_stat.csv and display the first few rows
history_stat = pd.read_csv('data/history_stat.csv')
history_stat.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [622]:
# Check years
history_stat.dtypes

date           object
home_team      object
away_team      object
home_score    float64
away_score    float64
tournament     object
city           object
country        object
neutral          bool
dtype: object

In [623]:
history_stat["date"] = pd.to_datetime(
    history_stat["date"],
    format="mixed",
    errors="coerce",
)

In [624]:
print("Earliest (oldest):", history_stat["date"].min())
print("Latest (newest):  ", history_stat["date"].max())
print("Oldest year:", history_stat["date"].min().year)
print("Newest year:", history_stat["date"].max().year)
print("Invalid dates:", history_stat["date"].isna().sum())

Earliest (oldest): 1872-11-30 00:00:00
Latest (newest):   2026-06-27 00:00:00
Oldest year: 1872
Newest year: 2026
Invalid dates: 0


In [625]:
history_stat.columns.to_list()

['date',
 'home_team',
 'away_team',
 'home_score',
 'away_score',
 'tournament',
 'city',
 'country',
 'neutral']

In [626]:
history_stat.describe()

,date,home_score,away_score
count,49287,49215.000000,49215.000000
mean,1994-04-13 22:21:42.915576064,1.756091,1.182404
min,1872-11-30 00:00:00,0.000000,0.000000
25%,1980-07-27 00:00:00,1.000000,0.000000
50%,2000-06-18 00:00:00,1.000000,1.000000
75%,2013-06-08 00:00:00,2.000000,2.000000
max,2026-06-27 00:00:00,31.000000,21.000000
std,NaN,1.770617,1.401770


In [627]:
# Load former_names.csv and display the first few rows
former_names = pd.read_csv('data/former_names.csv')
former_names.head()

,current,former,start_date,end_date
0,Benin,Dahomey,1959-11-08,1975-11-30
1,Burkina Faso,Upper Volta,1960-04-14,1984-08-04
2,Curaçao,Netherlands Antilles,1957-03-03,2010-10-10
3,Czechoslovakia,Bohemia,1903-04-05,1919-01-01
4,Czechoslovakia,Bohemia and Moravia,1939-01-01,1945-05-01


In [628]:
former_names.columns.to_list()

['current', 'former', 'start_date', 'end_date']

In [629]:
# Load elo.csv and display the first few rows
elo = pd.read_csv('data/elo.csv', on_bad_lines="skip")
elo.head()

,Team,Elo
0,Spain,2165
1,Argentina,2113
2,France,2081
3,England,2020
4,Brazil,1984


In [630]:
elo.columns.to_list()

['Team', 'Elo']

In [631]:
# Check if any rows have former names
history_stat["home_team"].isin(former_names["former"]).sum()

0

In [632]:
history_stat["away_team"].isin(former_names["former"]).sum()

0

In [633]:
# Check if the names in the history_stat matched the names in the elo
history_teams = set(history_stat["home_team"]).union(set(history_stat["away_team"]))

elo_teams = set(elo["Team"])

common_history_elo = history_teams & elo_teams

print("Total history teams:", len(history_teams))
print("Total elo teams:", len(elo_teams))
print("Common teams:", len(common_history_elo))

Total history teams: 333
Total elo teams: 244
Common teams: 226


In [634]:
# Drop rows that have team names in home_team or away_team are not in elo
valid_teams = set(elo["Team"])

history_stat_clean = history_stat[
    history_stat["home_team"].isin(valid_teams) &
    history_stat["away_team"].isin(valid_teams)
].copy()

In [635]:
print("Original rows:", len(history_stat))
print("Clean rows:", len(history_stat_clean))
print("Dropped rows:", len(history_stat) - len(history_stat_clean))

Original rows: 49287
Clean rows: 44402
Dropped rows: 4885


In [636]:
all_WC26_Teams = set(group_fixtures["home_team"]).union(set(group_fixtures["away_team"]))

common = common_history_elo & all_WC26_Teams

print("Total all_WC26_Teams:", len(all_WC26_Teams))
print("Total elo teams:", len(elo_teams))
print("Common teams:", len(common))

Total all_WC26_Teams: 48
Total elo teams: 244
Common teams: 41


In [637]:
missing_in_WC26 = all_WC26_Teams - elo_teams

list(missing_in_WC26)

['USA',
 'UEFA Playoff A',
 'FIFA Playoff 1',
 'UEFA Playoff B',
 'FIFA Playoff 2',
 'UEFA Playoff D',
 'UEFA Playoff C']

In [638]:
replaced_name = {
    "United States": "USA"
}

history_stat_clean["home_team"] = history_stat_clean["home_team"].replace(replaced_name)

history_stat_clean["away_team"] = history_stat_clean["away_team"].replace(replaced_name)

elo["Team"] = elo["Team"].replace(replaced_name)

In [639]:
#double check
elo_teams = set(elo["Team"])

missing_in_WC26 = all_WC26_Teams - elo_teams

list(missing_in_WC26)

['UEFA Playoff A',
 'FIFA Playoff 1',
 'UEFA Playoff B',
 'FIFA Playoff 2',
 'UEFA Playoff D',
 'UEFA Playoff C']

In [640]:
PLAYOFF_TO_TEAM = {
    "UEFA Playoff A": "Bosnia and Herzegovina",
    "UEFA Playoff B": "Sweden",
    "UEFA Playoff C": "Turkey",
    "UEFA Playoff D": "Czechia",
    "FIFA Playoff 1": "DR Congo",
    "FIFA Playoff 2": "Iraq",
}

In [641]:
def resolve_team(name):
    return PLAYOFF_TO_TEAM.get(name, name)

In [642]:
gf = group_fixtures.copy()
gf["home_team"] = gf["home_team"].map(resolve_team)
gf["away_team"] = gf["away_team"].map(resolve_team)

In [643]:
gf.head()

,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,Czechia,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,Bosnia and Herzegovina,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,Turkey,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [644]:
all_WC26_Teams_updated = set(gf["home_team"]).union(set(gf["away_team"]))
missing_in_WC26 = all_WC26_Teams_updated - elo_teams
missing_in_WC26

set()

In [645]:
# Combine csv files - elo.csv, history_stat.csv
elo_lookup = (
    elo.rename(columns={"Team": "team", "Elo": "elo"})
    .drop_duplicates(subset="team", keep="first")
)
history_with_elo = history_stat_clean.copy()
history_with_elo = history_with_elo.merge(
    elo_lookup.rename(columns={"team": "home_team", "elo": "home_elo"}),
    on="home_team",
    how="left",
)
history_with_elo = history_with_elo.merge(
    elo_lookup.rename(columns={"team": "away_team", "elo": "away_elo"}),
    on="away_team",
    how="left",
)

history_with_elo["elo_diff"] = history_with_elo["home_elo"] - history_with_elo["away_elo"]
print(history_with_elo.shape)
history_with_elo.head()

(44402, 12)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,away_elo,elo_diff
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1767,2020,-253
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,2020,1767,253
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1767,2020,-253
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2020,1767,253
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1767,2020,-253


In [646]:
main_cols = [
    "date", "home_team", "away_team", "home_score", "away_score", "tournament", "neutral",
    "home_elo", "away_elo", "elo_diff"
]
raw_cols = [ # This will be used to check history games before coming to the next match
    "date", "home_team", "away_team", "home_score", "away_score", "tournament"
]

mydf = history_with_elo[main_cols].copy()
rawdf = history_with_elo[raw_cols].copy()

# Do preprocessing for rawdf
rawdf["date"] = pd.to_datetime(rawdf["date"])

rawdf = rawdf.sort_values("date").reset_index(drop=True)

In [647]:
from collections import defaultdict

team_hist = defaultdict(list)

for r in rawdf.itertuples(index=False):
    team_hist[r.home_team].append(r)
    team_hist[r.away_team].append(r)

In [648]:
# Check row having missing values
mydf[mydf.isna().any(axis=1)].shape[0]

69

In [649]:
mydf = mydf.dropna()

In [650]:
# Create home advantage
mydf["home_advantage"] = mydf["neutral"].apply(lambda x: 0 if x == 1 else 1)

In [651]:
import numpy as np

TEAM_DISCIPLINE = {
    "Argentina": 2.4,
    "Saudi Arabia": 4.7,
    "Serbia": 4.0,
    "Netherlands": 2.8,
    "Switzerland": 2.3,
    "Ghana": 2.7,
    "Morocco": 1.6,
    "France": 1.1,
    "Croatia": 1.1,
    "Uruguay": 2.7,
    "Canada": 2.7,
    "Qatar": 2.3,
    "Cameroon": 3.3,
    "Senegal": 1.8,
    "Australia": 1.8,
    "Poland": 1.8,
    "Iran": 2.3,
    "Mexico": 2.3,
    "Japan": 1.5,
    "Portugal": 1.2,
    "South Korea": 1.5,
    "Costa Rica": 2.0,
    "Brazil": 1.2,
    "United States": 1.3,
    "Tunisia": 1.7,
    "Wales": 2.7,
    "Denmark": 1.7,
    "Belgium": 1.7,
    "Germany": 1.0,
    "Ecuador": 1.0,
    "Spain": 0.5,
    "England": 0.2
}

DEFAULT_DISC = np.mean(list(TEAM_DISCIPLINE.values()))

In [652]:
def team_disc(team):
    return TEAM_DISCIPLINE.get(team, DEFAULT_DISC)

In [653]:
def tournament_weight(tournament):
    if pd.isna(tournament):
        return 1.0

    t = str(tournament).lower()

    if "friendly" in t:
        return 0.7
    elif "world cup" in t:
        return 1.3
    else:
        return 1.0

In [654]:
def get_team_rates(team_hist, team, current_date, window=10, default=1.0):

    history = team_hist.get(team, [])

    # filter only relevant games
    history = [h for h in history if h.date < current_date]

    if len(history) == 0:
        return default, default

    history = history[-window:]

    scored, conceded = [], []

    for h in history:
        if h.home_team == team:
            scored.append(h.home_score)
            conceded.append(h.away_score)
        else:
            scored.append(h.away_score)
            conceded.append(h.home_score)

    weights = np.array([tournament_weight(h.tournament) for h in history])

    n = len(history)

    # =========================
    # CASE 1: 1–5 games
    # =========================
    if n <= 5:

        gs = np.sum(np.array(scored) * weights) / np.sum(weights)
        gc = np.sum(np.array(conceded) * weights) / np.sum(weights)

        alpha = n / 5

        return (
            alpha * gs + (1 - alpha) * default,
            alpha * gc + (1 - alpha) * default
        )

    # =========================
    # CASE 2: 6–10 games
    # =========================
    last5 = history[-5:]
    prev5 = history[-10:-5]

    def calc(chunk):
        if len(chunk) == 0:
            return default, default

        s, c, w = [], [], []

        for h in chunk:
            if h.home_team == team:
                s.append(h.home_score)
                c.append(h.away_score)
            else:
                s.append(h.away_score)
                c.append(h.home_score)

            w.append(tournament_weight(h.tournament))

        s = np.array(s)
        c = np.array(c)
        w = np.array(w)

        return np.sum(s * w) / np.sum(w), np.sum(c * w) / np.sum(w)

    A5, D5 = calc(last5)
    A10, D10 = calc(prev5)

    attack = 0.7 * A5 + 0.3 * A10
    defense = 0.7 * D5 + 0.3 * D10

    return attack, defense

In [655]:
def team_attack(team_hist, team, current_date):
    attack_rate, _ = get_team_rates(team_hist, team, current_date)
    return attack_rate

def team_def(team_hist, team, current_date):
    _, def_rate = get_team_rates(team_hist, team, current_date)
    return def_rate

In [656]:
# =========================
# YELLOW CARDS
# =========================
def pseudo_yellow(team_hist, row):

    h_disc = team_disc(row["home_team"])
    a_disc = team_disc(row["away_team"])

    h_attack = team_attack(team_hist, row["home_team"], row["date"])
    a_attack = team_attack(team_hist, row["away_team"], row["date"])

    # match intensity
    intensity = 0.6 * (h_attack + a_attack)

    # discipline increases fouls/cards
    disc = h_disc + a_disc

    base = intensity + 0.9 * disc

    # tournament boost
    t = str(row.get("tournament", "")).lower()
    if "world cup" in t:
        base *= 1.2
    elif "friendly" in t:
        base *= 0.9

    rng = np.random.default_rng(hash(row.name) % 2**32)

    return rng.poisson(np.clip(base, 0.5, 7))


# =========================
# RED CARDS
# =========================
def pseudo_red(team_hist, row):

    h_disc = team_disc(row["home_team"])
    a_disc = team_disc(row["away_team"])

    h_attack = team_attack(team_hist, row["home_team"], row["date"])
    a_attack = team_attack(team_hist, row["away_team"], row["date"])

    intensity = abs(h_attack - a_attack)

    disc_risk = h_disc + a_disc

    p = 0.01 + 0.008 * intensity + 0.012 * disc_risk

    t = str(row.get("tournament", "")).lower()
    if "world cup" in t:
        p += 0.01

    p = np.clip(p, 0.005, 0.25)

    rng = np.random.default_rng(hash(row.name) % 2**32)

    return int(rng.random() < p)


# =========================
# CORNERS
# =========================
def pseudo_corners(team_hist, row):

    h_attack = team_attack(team_hist, row["home_team"], row["date"])
    a_attack = team_attack(team_hist, row["away_team"], row["date"])

    h_def = team_def(team_hist, row["home_team"], row["date"])
    a_def = team_def(team_hist, row["away_team"], row["date"])

    # pressure model (attack vs opponent defense)
    home_pressure = h_attack + (1.0 - a_def)
    away_pressure = a_attack + (1.0 - h_def)

    base_home = 4.5 + 1.0 * (home_pressure - away_pressure)
    base_away = 4.5 + 1.0 * (away_pressure - home_pressure)

    # clamp realistic ranges
    base_home = np.clip(base_home, 1.0, 12)
    base_away = np.clip(base_away, 1.0, 12)

    rng = np.random.default_rng(hash(row.name) % 2**32)

    home_corners = rng.poisson(base_home)
    away_corners = rng.poisson(base_away)

    return home_corners, away_corners

In [683]:
def add_match_features(mydf, team_hist, window=10):

    mydf = mydf.copy()

    home_attack, home_defense = [], []
    away_attack, away_defense = [], []

    home_disc, away_disc = [], []

    yellow_home, yellow_away = [], []
    red_home, red_away = [], []

    home_corners_list, away_corners_list = [], []

    for _, row in mydf.iterrows():

        date = row["date"]

        h_team = row["home_team"]
        a_team = row["away_team"]

        # =========================
        # ATTACK / DEFENSE
        # =========================
        ha, hd = get_team_rates(team_hist, h_team, date, window)
        aa, ad = get_team_rates(team_hist, a_team, date, window)

        home_attack.append(ha)
        home_defense.append(hd)
        away_attack.append(aa)
        away_defense.append(ad)

        # =========================
        # DISCIPLINE
        # =========================
        h_d = team_disc(h_team)
        a_d = team_disc(a_team)

        home_disc.append(h_d)
        away_disc.append(a_d)

        # =========================
        # YELLOW CARDS
        # =========================
        y_home = pseudo_yellow(team_hist, row)
        y_away = pseudo_yellow(team_hist, row)

        yellow_home.append(y_home)
        yellow_away.append(y_away)

        # =========================
        # RED CARDS
        # =========================
        r_home = pseudo_red(team_hist, row)
        r_away = pseudo_red(team_hist, row)

        red_home.append(r_home)
        red_away.append(r_away)

        # =========================
        # CORNERS
        # =========================
        hc, ac = pseudo_corners(team_hist, row)

        home_corners_list.append(hc)
        away_corners_list.append(ac)

    # =========================
    # ASSIGN TO DF
    # =========================
    mydf["home_attack_rate"] = home_attack
    mydf["home_defense_rate"] = home_defense
    mydf["away_attack_rate"] = away_attack
    mydf["away_defense_rate"] = away_defense

    mydf["home_disc"] = home_disc
    mydf["away_disc"] = away_disc
    mydf["disc_diff"] = mydf["home_disc"] - mydf["away_disc"]
    mydf["disc_sum"] = mydf["home_disc"] + mydf["away_disc"]

    mydf["home_yellow"] = yellow_home
    mydf["away_yellow"] = yellow_away

    mydf["home_red"] = red_home
    mydf["away_red"] = red_away

    mydf["home_corners"] = home_corners_list
    mydf["away_corners"] = away_corners_list

    print("################ DONE WITH ADDING FEATURES TO DATAFRAME ####################")

    return mydf

In [658]:
# Split data into 3 parts (train, validate, test)
# Note: not split randomly, train 1872 - 2018; validate 2019 - 2024; test 2025 - 2026

train_end = pd.Timestamp("2018-12-31")
val_start = pd.Timestamp("2019-01-01")
val_end   = pd.Timestamp("2024-12-31")
test_start = pd.Timestamp("2025-01-01")

mydf_train = mydf[mydf["date"] <= train_end].copy()
mydf_val   = mydf[(mydf["date"] >= val_start) & (mydf["date"] <= val_end)].copy()
mydf_test  = mydf[mydf["date"] >= test_start].copy()

print("train:", mydf_train.shape, mydf_train["date"].min(), "→", mydf_train["date"].max())
print("val:  ", mydf_val.shape,   mydf_val["date"].min(),   "→", mydf_val["date"].max())
print("test: ", mydf_test.shape,  mydf_test["date"].min(),  "→", mydf_test["date"].max())

train: (37773, 11) 1872-11-30 00:00:00 → 2018-12-31 00:00:00
val:   (5463, 11) 2019-01-02 00:00:00 → 2024-12-31 00:00:00
test:  (1097, 11) 2025-01-02 00:00:00 → 2026-03-31 00:00:00


In [661]:
mydf_train = add_match_features(mydf_train, team_hist).copy()

################ DONE WITH ADDING FEATURES ####################


In [673]:
mydf_train.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'neutral', 'home_elo', 'away_elo', 'elo_diff',
       'home_advantage', 'home_attack_rate', 'home_defense_rate',
       'away_attack_rate', 'away_defense_rate', 'home_disc', 'away_disc',
       'disc_diff', 'disc_sum', 'home_yellow', 'away_yellow', 'home_red',
       'away_red', 'home_corners', 'away_corners'],
      dtype='object')

In [684]:
mydf_val = add_match_features(mydf_val, team_hist).copy()

################ DONE WITH ADDING FEATURES TO DATAFRAME ####################


In [685]:
goal_cols = ["home_elo", "away_elo", "elo_diff", "home_advantage", "home_attack_rate", "home_defense_rate", "away_attack_rate", "away_defense_rate"]
card_cols = goal_cols + ["home_disc", "away_disc", "disc_diff", "disc_sum"]

X_goals_train = mydf_train[goal_cols]
X_cards_train = mydf_train[card_cols]
y_home_goals_train = mydf_train["home_score"]
y_away_goals_train = mydf_train["away_score"]
y_home_yellow_train = mydf_train["home_yellow"]
y_away_yellow_train = mydf_train["away_yellow"]
y_home_red_train = mydf_train["home_red"]
y_away_red_train = mydf_train["away_red"]
y_home_corners_train = mydf_train["home_corners"]
y_away_corners_train = mydf_train["away_corners"]

X_goals_val = mydf_val[goal_cols]
X_cards_val = mydf_val[card_cols]
y_home_goals_val = mydf_val["home_score"]
y_away_goals_val = mydf_val["away_score"]
y_home_yellow_val = mydf_val["home_yellow"]
y_away_yellow_val = mydf_val["away_yellow"]
y_home_red_val = mydf_val["home_red"]
y_away_red_val = mydf_val["away_red"]
y_home_corners_val = mydf_val["home_corners"]
y_away_corners_val = mydf_val["away_corners"]


# X_goals_test = mydf_test[goal_cols]
# X_cards_test = mydf_test[card_cols]
# y_home_goals_test = mydf_test["home_score"]
# y_away_goals_test = mydf_test["away_score"]
# y_home_yellow_test = mydf_test["home_yellow"]
# y_away_yellow_test = mydf_test["away_yellow"]
# y_home_red_test = mydf_test["home_red"]
# y_away_red_test = mydf_test["away_red"]
# y_home_corners_test = mydf_test["home_corners"]
# y_away_corners_test = mydf_test["away_corners"]

In [672]:
print("X_goals_train:", X_goals_train.shape)
print("X_cards_train:", X_cards_train.shape)

print("y_home_goals_train:", y_home_goals_train.shape)
print("y_away_goals_train:", y_away_goals_train.shape)

print("y_home_yellow_train:", y_home_yellow_train.shape)
print("y_away_yellow_train:", y_away_yellow_train.shape)

print("y_home_red_train:", y_home_red_train.shape)
print("y_away_red_train:", y_away_red_train.shape)

print("y_home_corners_train:", y_home_corners_train.shape)
print("y_away_corners_train:", y_away_corners_train.shape)

X_goals_train: (37773, 8)
X_cards_train: (37773, 12)
y_home_goals_train: (37773,)
y_away_goals_train: (37773,)
y_home_yellow_train: (37773,)
y_away_yellow_train: (37773,)
y_home_red_train: (37773,)
y_away_red_train: (37773,)
y_home_corners_train: (37773,)
y_away_corners_train: (37773,)


## **Model Building**

- Goals: Poisson Regressor (x2) XGBoost regressor - count:poisson
- Yellow cards: Poisson Regressor (x2) LightGBM Regressor - tweedie or XGBoost regressor - count:poisson
- Red cards: Logistic regression (x2)
- Corners:	Poisson Regressor (x2) LightGBM Regressor - regression

In [675]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
from lightgbm import LGBMRegressor

### **Goal models**
- XGBoost Possion

In [686]:
X_goals_train_full = pd.concat(
    [X_goals_train, X_goals_val],
    axis=0
)

X_goals_train_full = X_goals_train_full.loc[:, goal_cols]

In [ ]:
y_home_goals_train_full = pd.concat(
    [y_home_goals_train, y_home_goals_val],
    axis=0
)

y_away_goals_train_full = pd.concat(
    [y_away_goals_train, y_away_goals_val],
    axis=0
)

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

param_grid_xgb = {
    "objective": ["count:poisson", "reg:squarederror"],
    "n_estimators": [200, 400],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "gamma": [0, 0.1]
}

xgb_home_goal_model = xgb.XGBRegressor(random_state=42)

search_home = GridSearchCV(
    estimator=xgb_home_goal_model,
    param_grid=param_grid_xgb,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    verbose=2,
    n_jobs=-1
)

xgb_away_goal_model = xgb.XGBRegressor(random_state=42)

search_away = GridSearchCV(
    estimator=xgb_away_goal_model,
    param_grid=param_grid_xgb,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    verbose=2,
    n_jobs=-1
)

In [ ]:
# def eval_model(y_pred, y_test, sim_n=2000, show_table=True):

#     y_pred = np.array(y_pred)
#     y_test = np.array(y_test)

#     # --- regression metrics ---
#     mae = mean_absolute_error(y_test, y_pred)
#     rmse = np.sqrt(mean_squared_error(y_test, y_pred))

#     # --- true results ---
#     def result(h, a):
#         if h > a:
#             return "H"
#         elif a > h:
#             return "A"
#         return "D"

#     true_res = [result(h, a) for h, a in y_test]

#     # =========================
#     # ⚽ simulation-based prediction (MAIN LOGIC)
#     # =========================
#     sim_pred_res = []
#     sim_best_scores = []

#     for i in range(len(y_pred)):

#         sim = simulate_score(y_pred[i, 0], y_pred[i, 1], n_sim=sim_n)

#         score_counts = Counter(sim)
#         best_score = score_counts.most_common(1)[0][0]

#         sim_best_scores.append(best_score)

#         h, a = best_score
#         if h > a:
#             sim_pred_res.append("H")
#         elif a > h:
#             sim_pred_res.append("A")
#         else:
#             sim_pred_res.append("D")

#     # --- accuracy ---
#     sim_match_acc = np.mean(np.array(sim_pred_res) == np.array(true_res))

#     # --- score accuracy ---
#     home_score_acc = np.mean(np.round(y_pred[:, 0]) == y_test[:, 0])
#     away_score_acc = np.mean(np.round(y_pred[:, 1]) == y_test[:, 1])

#     sim_score_acc = np.mean(
#         np.array(sim_best_scores) == list(zip(y_test[:, 0], y_test[:, 1]))
#     )

#     # --- table ---
#     df = pd.DataFrame({
#         "home_actual": y_test[:, 0],
#         "away_actual": y_test[:, 1],
#         "home_pred": np.round(y_pred[:, 0], 2),
#         "away_pred": np.round(y_pred[:, 1], 2),

#         "sim_home_pred": [s[0] for s in sim_best_scores],
#         "sim_away_pred": [s[1] for s in sim_best_scores],

#         "actual_result": true_res,
#         "sim_result": sim_pred_res
#     })

#     if show_table:
#         display(df.head(15))

#     return {
#         "MAE": mae,
#         "RMSE": rmse,
#         "Home Score Accuracy": home_score_acc,
#         "Away Score Accuracy": away_score_acc,
#         "Simulation Match Accuracy": sim_match_acc
#     }

In [ ]:
search_home.fit(X_goals_train_full, y_home_goals_train_full)
search_away.fit(X_goals_train_full, y_away_goals_train_full)

Fitting 5 folds for each of 384 candidates, totalling 1920 fits
Fitting 5 folds for each of 384 candidates, totalling 1920 fits


GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None...
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 1.0], 'gamma': [0, 0.1],
                         'learning_rate': [0.01, 0.05], 'max_depth': [3, 4, 5],
                         'min_child_weight': [1, 5], 'n_estimators': [200, 400],
                         'objective': ['count:poisson', 'reg:squarederror'],
                         'subsample': [0.8, 1.0]},
             scoring='neg_mean_absolute_error', verbose=2)

In [ ]:
best_home_model = search_home.best_estimator_
best_away_model = search_away.best_estimator_

### **Yellow Card models**

### **Red Card models**

### **Corner models**

## **Match Simulation Setup**
- simulate_match
- group_simulate
- knockout_simulate
- wc_simulate
- monte_carlo_simulate